# 04 — SDK Hello World: Your First Agent

**Goal:** Write and run your first OpenHands agent using the Python SDK — from LLM setup to autonomous code execution.

**What You'll Learn:**
- The three core SDK objects: LLM, Agent, Conversation
- How to configure tools for your agent
- Understanding the event stream
- Running agents locally vs remotely
- Interpreting agent output and exit conditions


## 4.1 The Three Core Objects

Every OpenHands SDK program uses exactly three objects:

```
LLM ────────────→ Agent ────────────→ Conversation
(brain)          (capabilities)     (execution loop)
```

| Object | Role | Key Parameters |
|---|---|---|
| **LLM** | The language model that does the reasoning | `model`, `api_key`, `base_url` |
| **Agent** | Defines what the agent can do — which tools, which system prompt | `llm`, `tools`, `system_prompt` |
| **Conversation** | The execution loop — holds event history, drives iteration | `agent`, `workspace` |

**The flow:** You create an LLM → wrap it in an Agent with tools → create a Conversation that runs the agent loop in a workspace.


## 4.2 Minimal Working Example

This is the smallest possible OpenHands agent — it creates a file with facts about the current project.


In [ ]:
# ═══════════════════════════════════════════
# 4.2 Minimal Working Agent
# ═══════════════════════════════════════════
import os
from pathlib import Path

# ─── Step 1: Create the LLM (the brain) ───
# The LLM object wraps the model API — it handles auth, requests, and responses
# All LLM calls go through litellm, so any supported provider works the same way
from openhands.sdk import LLM

llm = LLM(
    model=os.getenv('LLM_MODEL', 'gpt-5.5'),  # Model identifier (litellm format)
    api_key=os.getenv('LLM_API_KEY'),           # NEVER hardcode keys — use env vars
)
print(f'LLM created: model={llm.model}')

# ─── Step 2: Create the Agent (the capabilities) ───
# The Agent ties together: which LLM to use + which tools it has access to
# Tools define the agent's action space — what it can DO in the world
from openhands.sdk import Agent
from openhands.tools.file_editor import FileEditorTool
from openhands.tools.terminal import TerminalTool
from openhands.tools.task_tracker import TaskTrackerTool
from openhands.tools import Tool

# Wrap tool classes in Tool() — this registers them with the agent system
# Each Tool specifies: name, what it does, and how to execute it
agent = Agent(
    llm=llm,
    tools=[
        Tool(name=FileEditorTool.name),   # Read/write/edit files
        Tool(name=TerminalTool.name),     # Execute bash commands
        Tool(name=TaskTrackerTool.name),  # Track subtask progress
    ],
)
print(f'Agent created with {len(agent.tools)} tools')

# ─── Step 3: Create the Conversation (the execution loop) ───
# The Conversation drives the agent loop: send message → agent reasons →
# agent acts → observe result → repeat until finished
# The 'workspace' is where file operations happen — here it's a temp directory
from openhands.sdk import Conversation
import tempfile

# Create a temporary workspace — the agent's "desk" where it can create/edit files
workspace_dir = tempfile.mkdtemp(prefix='openhands_demo_')
print(f'Workspace: {workspace_dir}')

conversation = Conversation(
    agent=agent,
    workspace=workspace_dir,  # Agent can read/write files here
)

# ─── Step 4: Send a task and run ───
# This is the key method: starts the agent loop and blocks until completion
# The agent will: plan → execute tools → observe → iterate → finish
task = "Write 3 facts about Python into a file called FACTS.txt"
print(f'Task: {task}')
conversation.send_message(task)
conversation.run()  # This is the blocking call — agent works autonomously here

# ─── Step 5: Check results ───
# After run() returns, inspect what the agent created
facts_file = Path(workspace_dir) / 'FACTS.txt'
if facts_file.exists():
    print(f'\n✓ Agent created FACTS.txt:')
    print('─' * 40)
    print(facts_file.read_text())
    print('─' * 40)
else:
    print('✗ FACTS.txt was not created — check API key and model access')

# Clean up temporary workspace
import shutil
shutil.rmtree(workspace_dir, ignore_errors=True)
print(f'\nCleaned up workspace: {workspace_dir}')


## 4.3 Understanding the Event Stream

Every agent action generates events. Understanding this stream is crucial for debugging and custom applications.

**Event types:**
- `MessageEvent` — User message or agent response
- `ActionEvent` — Agent proposes an action (write file, run bash, etc.)
- `ObservationEvent` — Result of executing an action
- `StateChangeEvent` — Internal state transitions
- `FinishEvent` — Agent declares task complete


In [ ]:
# ═══════════════════════════════════════════
# 4.3 Reading the Event Stream
# ═══════════════════════════════════════════
# The conversation object stores all events in chronological order.
# You can inspect them to understand what the agent did step by step.

# After conversation.run() completes, events are accessible:
# (This is a DEMONSTRATION — run the cell above first to populate events)

from collections import Counter

print('Event Stream Analysis')
print('=' * 60)

# Count events by type to see the agent's activity profile
# This tells you: did it spend more time editing files or running commands?
try:
    events = conversation.events  # All events from the completed run
    type_counts = Counter(type(e).__name__ for e in events)
    print(f'Total events: {len(events)}')
    for event_type, count in type_counts.most_common():
        print(f'  {event_type}: {count}')

    # Show the last few events — these reveal how the agent concluded
    print(f'\nLast 3 events (the conclusion):')
    for event in events[-3:]:
        # Each event has a message/content field with human-readable info
        msg = getattr(event, 'message', None) or getattr(event, 'content', str(event)[:80])
        print(f'  [{type(event).__name__}] {msg}')
except NameError:
    print('No conversation object available — run the cell in 4.2 first')
except Exception as e:
    print(f'Error reading events: {e}')


## 4.4 Agent with Custom System Prompt

You can override the default system prompt to give the agent a specific personality or domain expertise.


In [ ]:
# ═══════════════════════════════════════════
# 4.4 Custom System Prompt Agent
# ═══════════════════════════════════════════
# The system prompt is the "constitution" of the agent — it defines its role,
# constraints, and behavior. Custom prompts let you specialize agents.

custom_prompt = """You are a Python code reviewer. Your job is to:
1. Read the provided Python files carefully
2. Identify bugs, security issues, and style violations
3. Write your findings to REVIEW.md
4. NEVER modify the original files — only write to REVIEW.md
5. Be thorough but concise — prioritize critical over cosmetic issues
"""

# Create agent with custom system prompt — overrides the default coding agent prompt
reviewer_agent = Agent(
    llm=llm,
    tools=[
        Tool(name=FileEditorTool.name),   # To read source files and write REVIEW.md
        Tool(name=TerminalTool.name),     # To run the code and see errors
    ],
    system_prompt=custom_prompt,          # This replaces the default prompt
)

print('Reviewer agent created with custom prompt')
print(f'System prompt length: {len(custom_prompt)} chars')
print(f'Key constraint: agent will only write to REVIEW.md, not modify source files')


## 4.5 Error Handling and Agent Limits

Agents can fail, loop infinitely, or produce bad output. The SDK provides controls:

| Control | Default | Purpose |
|---|---|---|
| `max_iterations` | 100 | Max agent loop cycles before forced stop |
| `timeout` | None | Wall-clock timeout for the entire run |
| `condenser` | Auto | Compresses history when context gets too long |
| Security Analyzer | Enabled | Blocks high-risk actions |


In [ ]:
# ═══════════════════════════════════════════
# 4.5 Agent with Limits
# ═══════════════════════════════════════════
# Production agents should always have limits — infinite loops waste API credits

# The Conversation object accepts limits that prevent runaway agents
# These are safety nets, not targets — a good agent finishes well before limits
from openhands.sdk import Conversation as LimitedConversation

# Create a conversation with explicit limits
limited_conv = Conversation(
    agent=agent,
    workspace=workspace_dir,
    max_iterations=20,     # Stop after 20 reasoning-action cycles
    # timeout=300,         # Or stop after 5 minutes (wall clock)
)

print('Conversation limits configured:')
print(f'  Max iterations: {getattr(limited_conv, "max_iterations", "default (100)")}')
print(f'  Tip: set max_iterations lower during development to catch loops early')
print(f'  Tip: use timeout for production to guarantee termination')


## 4.6 Key Takeaways

1. **LLM → Agent → Conversation** is the universal pattern
2. **Tools are the action space** — without tools, the agent can only talk
3. **The workspace is the agent's sandbox** — everything happens inside it
4. **Events are the source of truth** — inspect them to debug agent behavior
5. **Always set limits** — max_iterations and timeouts prevent runaway costs


## Next

→ [05_tools_and_workspace.ipynb](05_tools_and_workspace.ipynb) — Deep dive into all built-in tools and workspace types
